# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# ProbSS 3 — Exploring data without hiding uncertainty

## What you will do

First decide what quantity you want to estimate. Then calculate a Hoeffding
interval, check its assumptions against a plot, and explain clearly which
population the result may describe.

The automobile table is a course copy with no documented source or sampling
frame. That missing information is itself something the analysis must report.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path

def course_data(filename):
    candidates = (
        Path("data") / filename,
        Path("master/jp/data") / filename,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from master/jp "
        "or from the repository root."
    )


## 1. Say what you want to estimate and what you assume

For each row define $Y=1$ if fuel economy is at least 30 miles per gallon and $Y=0$ otherwise. The sample proportion estimates $p=P(Y=1)$ only after a target population and sampling mechanism are specified.

If the rows were independent and identically distributed observations from that target, Hoeffding's inequality would give the two-sided interval

$$
\widehat p_n \pm \sqrt{\frac{\log(2/\alpha)}{2n}},
$$

clipped to $[0,1]$. Boundedness is satisfied because $Y$ is binary. Independence, identical distribution, and representativeness must be checked rather than assumed.


In [ ]:
auto_path = course_data("auto.csv")
auto = pd.read_csv(auto_path)
for column in auto.columns:
    auto[column] = pd.to_numeric(auto[column], errors="coerce")

quality = pd.DataFrame(
    {
        "missing": auto.isna().sum(),
        "minimum": auto.min(numeric_only=True),
        "maximum": auto.max(numeric_only=True),
    }
)
print("Rows and columns:", auto.shape)
quality


In [ ]:
alpha = 0.05
efficient = (auto["mpg"] >= 30).astype(float)
n = len(efficient)
p_hat = efficient.mean()
radius = np.sqrt(np.log(2 / alpha) / (2 * n))
interval = (max(0.0, p_hat - radius), min(1.0, p_hat + radius))

print(f"Sample proportion: {p_hat:.3f}")
print(f"Nominal 95% Hoeffding interval under IID sampling: {interval}")


The word nominal is important. The calculation is algebraically correct under the assumptions, but it is not automatically a confidence statement about all cars.


## 2. Look for patterns an IID model misses


In [ ]:
ordered = auto.sort_values("model-year").reset_index(drop=True)
ordered["efficient"] = (ordered["mpg"] >= 30).astype(float)
ordered["running_rate"] = ordered["efficient"].expanding().mean()
by_year = ordered.groupby("model-year")["efficient"].agg(["mean", "count"])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(ordered["model-year"], ordered["running_rate"])
axes[0].axhline(p_hat, color="black", linestyle="--")
axes[0].set(
    xlabel="model year",
    ylabel="running sample proportion",
    title="Running rate in the file's year order",
)

axes[1].scatter(
    by_year.index,
    by_year["mean"],
    s=4 * by_year["count"],
)
axes[1].set(
    xlabel="model year",
    ylabel="proportion with mpg at least 30",
    title="Composition changes across model years",
    ylim=(-0.05, 1.05),
)
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
assumption_audit = pd.DataFrame(
    {
        "assumption": [
            "The statistic is bounded in [0, 1]",
            "Rows are independent",
            "Rows have one common distribution",
            "The sample represents a named population",
            "The source and reuse terms are documented",
        ],
        "evidence": [
            "Yes, by construction",
            "Not documented; related vehicle models may occur",
            "Questionable: the year diagnostic changes strongly",
            "No sampling frame is supplied",
            "No: provenance must be resolved before publication",
        ],
        "consequence": [
            "Hoeffding's boundedness condition is met",
            "The nominal interval may be too narrow",
            "One common p may not answer a useful question",
            "Do not generalise beyond this course copy",
            "Do not redistribute as an authoritative source",
        ],
    }
)
assumption_audit


In [ ]:
caption = (
    f"In the {n}-row course automobile table, {100*p_hat:.1f}% of rows have "
    "fuel economy of at least 30 mpg. Under an IID sampling model the nominal "
    f"95% Hoeffding interval is [{interval[0]:.3f}, {interval[1]:.3f}]. "
    "The proportion changes markedly by model year, and the file supplies no "
    "sampling frame, so the interval should not be presented as population "
    "coverage for all cars."
)
print(caption)


## Recap

Before you finish, make sure you can:

1. State the observation, target variable, target population, and sampling assumption.
2. Show the interval calculation and the diagnostic figure.
3. Decide whether the IID interval answers a reasonable scientific question. If not, propose a stratified or time-specific target.
4. Write a two-sentence caption for a nontechnical audience that reports both the numerical result and the limitation.
